In [4]:
!which python
import sys
sys.path

/opt/miniconda3/bin/python


['/Users/pandysudhan/dev/projects/nano_gpt',
 '/opt/miniconda3/envs/GPT/lib/python312.zip',
 '/opt/miniconda3/envs/GPT/lib/python3.12',
 '/opt/miniconda3/envs/GPT/lib/python3.12/lib-dynload',
 '',
 '/opt/miniconda3/envs/GPT/lib/python3.12/site-packages']

In [5]:
# We always start with a dataset to train on. Let's download the tiny shakespeare dataset
!wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt

--2024-04-13 11:08:26--  https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 2606:50c0:8000::154, 2606:50c0:8001::154, 2606:50c0:8002::154, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|2606:50c0:8000::154|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1115394 (1.1M) [text/plain]
Saving to: ‘input.txt.3’

input.txt.3         100%[===================>]   1.06M  2.39MB/s    in 0.4s    

2024-04-13 11:08:26 (2.39 MB/s) - ‘input.txt.3’ saved [1115394/1115394]



In [6]:
with open('input.txt', 'r', encoding='utf-8') as file:
    text = file.read()
print(len(text))

1115394


In [26]:
vocab_size = len(text)
embd_size = 128
device = 'cuda' if torch.cuda.is_available() else 'cpu'


In [7]:
print(text[:1000])

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us kill him, and we'll have corn at our own price.
Is't a verdict?

All:
No more talking on't; let it be done: away, away!

Second Citizen:
One word, good citizens.

First Citizen:
We are accounted poor citizens, the patricians good.
What authority surfeits on would relieve us: if they
would yield us but the superfluity, while it were
wholesome, we might guess they relieved us humanely;
but they think we are too dear: the leanness that
afflicts us, the object of our misery, is as an
inventory to particularise their abundance; our
sufferance is a gain to them Let us revenge this with
our pikes, ere we become rakes: for the gods know I
speak this in hunger for bread, not in thirst for revenge.



In [8]:
char_vocab = sorted(list(set(text)))
print("all chars in the text:", char_vocab)
print(len(char_vocab))

all chars in the text: ['\n', ' ', '!', '$', '&', "'", ',', '-', '.', '3', ':', ';', '?', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z']
65


# Encoder/decoder for characters with dictionary

In [9]:
def create_custom_char_encoder(custom_chars):
    char_to_index = {char: i for i, char in enumerate(custom_chars)} # creates dict {char: int}
    index_to_char = {i: char for char, i in char_to_index.items()} #creates dict {int: char}
    vocab_size = len(custom_chars)

    encode = lambda sentence: [char_to_index[char] for char in sentence if char in char_to_index] 
    decode = lambda indices: ''.join([index_to_char[i] for i in indices if i in index_to_char])

    return encode, decode


In [10]:
encode, decode = create_custom_char_encoder(char_vocab)

In [11]:
encode("hello")

[46, 43, 50, 50, 53]

In [12]:
decode([34,34])

'VV'

### Encode all the training data

In [22]:
import torch 
from torch.nn import functional as F
torch.manual_seed(8665)

In [14]:
data = torch.tensor(encode(text), dtype=torch.long)
print(data.shape, data.dtype)

torch.Size([1115394]) torch.int64


### Train test split


In [15]:
n = int(0.9 * len(data))
train_data= data[:n]
test_data = data[n:]

### Generating training data


In [16]:
block_size = 8 # also called context length 
batch_size = 4 # how many batches of training data to take at once (the target labels would be batch_size * block_size)

In [17]:
def generate_data_batch(split):
    data = train_data if split == "train" else test_data

    # get random indices to start training
    ix = torch.randint(len(data-block_size), (batch_size,))

    xb = torch.stack([data[i: i + block_size] for i in ix])
    yb = torch.stack([data[i+1: i + block_size+1] for i in ix])
        
    return xb, yb

In [18]:
generate_data_batch("train")

(tensor([[59, 50,  1, 58, 47, 51, 43,  1],
         [ 1, 39, 57,  1, 44, 47, 50, 50],
         [58, 11,  0, 18, 53, 56,  1, 35],
         [ 1, 58, 46, 53, 59,  1, 57, 46]]),
 tensor([[50,  1, 58, 47, 51, 43,  1, 53],
         [39, 57,  1, 44, 47, 50, 50, 57],
         [11,  0, 18, 53, 56,  1, 35, 39],
         [58, 46, 53, 59,  1, 57, 46, 53]]))

## Model part

In [31]:
class BigramModel(nn.Module):

    def __init__(self):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, embd_size)
        self.position_embedding_table = nn.Embedding(block_size, vocab_size)
        self.linear_embd_to_vocab = nn.Linear(embd_size, vocab_size)


    def forward(self, idx, targets=None):
        B, T = idx, shape
        token_embedding = self.token_embedding_table(idx)
        position_embedding_table = self.position_embedding_table(torch.arange(T, device=device))
        x = token_embedding + position_embedding_table
        logits = self.linear_embd_to_vocab(x)

        
        if targets == None:
            loss = None
        else:
            B,T,C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T,)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):

        for _ in max_new_tokens:
            idx_concated = idx[:, :-block_size, :]
            logits, loss = self(idx_concated)
        
            #convert last logits to probs
            logits = logits[:, -1, :]
            probs = F.softmax(logits, dim= -1)
    
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=-1)


        return idx

@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

In [34]:
learning_rate = 1e-3
max_iter = 100
eval_interval = 10

model = BigramModel()
model = model.to(device)

print("working on device: ", device, "\n")
print(sum(p.numel() for p in model.parameters())/1e6, 'M parameters')

optimizer = torch.optim.AdamW(model.parameters(), lr = learning_rate)


working on device:  cpu 

295.57941 M parameters


## Training loop

In [35]:
for iter in range(max_iter):
    if iter % eval_interval == 0 or iter == max_iter - 1:
        losses = estimate_loss()
        print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

    xb, yb = get_batch("train")

    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none = True)

    loss.backward()

    optimizer.step()

NameError: name 'eval_iters' is not defined